# Task 4: Implement Proposed Approach

In [1]:
!pip install -U transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling trans

In [2]:
import re, html, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, precision_score, recall_score


In [3]:
! mkdir -p data

# Download train/dev split file (practice split)
! wget -O data/practice_splits.zip \https://github.com/Perez-AlmendrosC/dontpatronizeme/archive/refs/heads/master.zip

# Download test set (raw TSV)
! wget -O data/task4_test.tsv \https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/TEST/task4_test.tsv

# Download dataset
! wget -O data/dontpatronizeme_pcl.tsv \https://raw.githubusercontent.com/CRLala/NLPLabs-2024/refs/heads/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv

# Unzip the dataset
!unzip data/practice_splits.zip -d data/

--2026-03-04 00:55:33--  https://github.com/Perez-AlmendrosC/dontpatronizeme/archive/refs/heads/master.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/Perez-AlmendrosC/dontpatronizeme/zip/refs/heads/master [following]
--2026-03-04 00:55:34--  https://codeload.github.com/Perez-AlmendrosC/dontpatronizeme/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 20.205.243.165
Connecting to codeload.github.com (codeload.github.com)|20.205.243.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘data/practice_splits.zip’

data/practice_split     [ <=>                ] 664.12K  --.-KB/s    in 0.09s   

2026-03-04 00:55:34 (7.05 MB/s) - ‘data/practice_splits.zip’ saved [680064]

--2026-03-04 00:55:34--  https://raw.githubusercontent.com/Perez

In [4]:
import pandas as pd

# Load the PCL Dataset TSV
full_df = pd.read_csv(
    "data/dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=3, # disclaimer is lines 1–3, data starts at line 4
    header=None
)

# Assign column names
full_df.columns = ["par_id", "art_id", "keyword", "country_code", "text", "label"]

# Convert graded label (0–4) -> binary (0/1) using {0,1}=NoPCL and {2,3,4}=PCL
full_df["label"] = full_df["label"].astype(int)
full_df["label_bin"] = (full_df["label"] >= 2).astype(int)

# Load split IDs
train_ids = pd.read_csv("data/dontpatronizeme-master/semeval-2022/practice splits/train_semeval_parids-labels.csv")
dev_ids   = pd.read_csv("data/dontpatronizeme-master/semeval-2022/practice splits/dev_semeval_parids-labels.csv")

# Ensure types match
full_df["par_id"] = full_df["par_id"].astype(int)
train_ids["par_id"] = train_ids["par_id"].astype(int)
dev_ids["par_id"] = dev_ids["par_id"].astype(int)

# Merge to get actual train/dev dataframes
train_df = full_df.merge(train_ids[["par_id"]], on="par_id", how="inner")
dev_df   = full_df.merge(dev_ids[["par_id"]], on="par_id", how="inner")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("\nTrain label distribution:\n", train_df["label_bin"].value_counts())

Train shape: (8375, 7)
Dev shape: (2094, 7)

Train label distribution:
 label_bin
0    7581
1     794
Name: count, dtype: int64


# Implementation: RoBERTa Baseline Model

In [5]:
def clean_text(s: str) -> str:
    s = str(s)
    s = html.unescape(s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"([!?.,])\1{2,}", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

train_df = train_df.copy()
dev_df = dev_df.copy()

train_df["text"] = train_df["text"].fillna("").astype(str).apply(clean_text)
dev_df["text"]   = dev_df["text"].fillna("").astype(str).apply(clean_text)

train_df["label_bin"] = train_df["label_bin"].astype(int)
dev_df["label_bin"]   = dev_df["label_bin"].astype(int)


In [6]:
# Build HF datasets from the DataFrames
from datasets import Dataset

train_hf = Dataset.from_pandas(train_df[["text", "label_bin"]].copy())
dev_hf   = Dataset.from_pandas(dev_df[["text", "label_bin"]].copy())

train_hf = train_hf.rename_column("label_bin", "labels")
dev_hf   = dev_hf.rename_column("label_bin", "labels")

In [7]:
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)

MAX_LEN = 128

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_hf = Dataset.from_pandas(train_df[["text", "label_bin"]].copy()).rename_column("label_bin", "labels")
dev_hf   = Dataset.from_pandas(dev_df[["text", "label_bin"]].copy()).rename_column("label_bin", "labels")

train_hf = train_hf.map(tokenize, batched=True)
dev_hf   = dev_hf.map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
train_hf.set_format(type="torch", columns=cols)
dev_hf.set_format(type="torch", columns=cols)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

In [8]:
# add sample weights for oversampling
y = train_df["label_bin"].to_numpy()
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=y)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

# per-sample weights for WeightedRandomSampler
sample_weights = np.where(y == 1, class_weights[1].item(), class_weights[0].item())
sample_weights = torch.tensor(sample_weights, dtype=torch.double)

In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.sample_weights = sample_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.sample_weights is None:
            return super().get_train_dataloader()
        sampler = WeightedRandomSampler(
            weights=self.sample_weights,
            num_samples=len(self.sample_weights),
            replacement=True
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=True
        )


In [11]:
# Metrics: F1 on positive class
import evaluate
import numpy as np

# metrics cell (earlier)
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=1)
    return {
        "f1_pos": f1_metric.compute(predictions=pred, references=labels, average="binary")["f1"],
        "precision_pos": precision_metric.compute(predictions=pred, references=labels, average="binary")["precision"],
        "recall_pos": recall_metric.compute(predictions=pred, references=labels, average="binary")["recall"],
    }


In [12]:
# Disabled on purpose to keep official dev clean for final evaluation.
# Use only the disciplined internal-split workflow in Cells 17-21.
print("Skipped: pre-tuning training/evaluation on official dev to avoid data leakage.")


Skipped: pre-tuning training/evaluation on official dev to avoid data leakage.


In [13]:
# Disabled on purpose to keep official dev clean for final evaluation.
# Threshold selection is done on internal dev in Cell 19 and applied once in Cell 21.
print("Skipped: threshold tuning on official dev to avoid data leakage.")


Skipped: threshold tuning on official dev to avoid data leakage.


## Hyperparameter Tuning

In [14]:
# Untouched until final evaluation
official_dev_df = dev_df.copy()


In [15]:
# Make internal split from the training data
from sklearn.model_selection import train_test_split

SEED = 42
train_internal_df, dev_internal_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label_bin"],
    random_state=SEED
)

print(train_internal_df["label_bin"].value_counts(normalize=True))
print(dev_internal_df["label_bin"].value_counts(normalize=True))


label_bin
0    0.90517
1    0.09483
Name: proportion, dtype: float64
label_bin
0    0.90533
1    0.09467
Name: proportion, dtype: float64


In [16]:
search_space = [
    {"lr": 2e-5, "wd": 0.01, "epochs": 4, "max_len": 128, "bs": 16},
    {"lr": 2e-5, "wd": 0.01, "epochs": 5, "max_len": 256, "bs": 16},
    {"lr": 3e-5, "wd": 0.01, "epochs": 4, "max_len": 128, "bs": 16},
    {"lr": 2e-5, "wd": 0.05, "epochs": 5, "max_len": 256, "bs": 16},
]

In [17]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, DataCollatorWithPadding

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

results = []

for cfg in search_space:
    print(f"Running config: {cfg}")

    # tokenizer for this max_len
    MAX_LEN = int(cfg["max_len"])
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

    # internal datasets
    tr_hf = Dataset.from_pandas(train_internal_df[["text","label_bin"]].copy()).rename_column("label_bin","labels")
    dv_hf = Dataset.from_pandas(dev_internal_df[["text","label_bin"]].copy()).rename_column("label_bin","labels")

    tr_hf = tr_hf.map(tokenize, batched=True)
    dv_hf = dv_hf.map(tokenize, batched=True)

    cols = ["input_ids","attention_mask","labels"]
    tr_hf.set_format(type="torch", columns=cols)
    dv_hf.set_format(type="torch", columns=cols)

    collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

    # class weights on internal train
    y_tr = train_internal_df["label_bin"].to_numpy()
    cw = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=y_tr)
    cw = torch.tensor(cw, dtype=torch.float32)

    # model + trainer
    model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

    args = TrainingArguments(
        output_dir=f"./tmp_tune_lr{cfg['lr']}_wd{cfg['wd']}_ep{cfg['epochs']}_ml{cfg['max_len']}",
        eval_strategy="no",
        save_strategy="no",
        learning_rate=float(cfg["lr"]),
        weight_decay=float(cfg["wd"]),
        num_train_epochs=int(cfg["epochs"]),
        per_device_train_batch_size=int(cfg["bs"]),
        per_device_eval_batch_size=32,
        fp16=True,
        report_to="none",
        seed=SEED,
    )

    trainer = WeightedTrainer(
        class_weights=cw,
        sample_weights=None,
        model=model,
        args=args,
        train_dataset=tr_hf,
        eval_dataset=dv_hf,
        data_collator=collator,
    )

    trainer.train()

    # predictions on internal dev
    pred = trainer.predict(dv_hf)
    logits = torch.tensor(pred.predictions, dtype=torch.float32)
    y_true = pred.label_ids
    probs = torch.softmax(logits, dim=1).numpy()[:, 1]

    # metric at threshold 0.5
    y_05 = (probs >= 0.5).astype(int)
    f1_at_05 = f1_score(y_true, y_05, zero_division=0)

    # best threshold on internal dev
    best_t, best_f1, best_p, best_r = 0.5, -1, 0.0, 0.0
    for t in np.linspace(0.05, 0.95, 181):
        y_hat = (probs >= t).astype(int)
        f1v = f1_score(y_true, y_hat, zero_division=0)
        if f1v > best_f1:
            best_f1 = f1v
            best_t = float(t)
            best_p = precision_score(y_true, y_hat, zero_division=0)
            best_r = recall_score(y_true, y_hat, zero_division=0)

    results.append({
        **cfg,
        "f1_05": f1_at_05,
        "best_t": best_t,
        "best_f1": best_f1,
        "best_p": best_p,
        "best_r": best_r,
    })

res_df = pd.DataFrame(results).sort_values("best_f1", ascending=False)
display(res_df)
best_cfg = res_df.iloc[0].to_dict()
best_cfg



Running config: {'lr': 2e-05, 'wd': 0.01, 'epochs': 4, 'max_len': 128, 'bs': 16}


Map:   0%|          | 0/7118 [00:00<?, ? examples/s]

Map:   0%|          | 0/1257 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.562417
1000,0.428867
1500,0.326077
2000,0.180281


,lr,wd,epochs,max_len,bs,f1_05,best_t,best_f1,best_p,best_r
3,0.00002,0.05,5,256,16,0.583333,0.355,0.586777,0.577236,0.596639
0,0.00002,0.01,4,128,16,0.566524,0.410,0.572650,0.582609,0.563025
1,0.00002,0.01,5,256,16,0.508929,0.050,0.553719,0.544715,0.563025
2,0.00003,0.01,4,128,16,0.483721,0.085,0.541833,0.515152,0.571429


{'lr': 2e-05,
 'wd': 0.05,
 'epochs': 5.0,
 'max_len': 256.0,
 'bs': 16.0,
 'f1_05': 0.5833333333333334,
 'best_t': 0.3549999999999999,
 'best_f1': 0.5867768595041323,
 'best_p': 0.5772357723577236,
 'best_r': 0.5966386554621849}

In [18]:
# Retrain on full train set (with locked best config)

MAX_LEN = int(best_cfg["max_len"])
BEST_T = float(best_cfg["best_t"])   # from internal-dev tuning only

# Build full-train + official-dev HF datasets
train_final_df = train_df.copy()
dev_final_df = official_dev_df.copy()   # official dev held out for final test

train_final_hf = Dataset.from_pandas(train_final_df[["text","label_bin"]].copy()).rename_column("label_bin","labels")
dev_final_hf   = Dataset.from_pandas(dev_final_df[["text","label_bin"]].copy()).rename_column("label_bin","labels")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_final_hf = train_final_hf.map(tokenize, batched=True)
dev_final_hf   = dev_final_hf.map(tokenize, batched=True)

cols = ["input_ids","attention_mask","labels"]
train_final_hf.set_format(type="torch", columns=cols)
dev_final_hf.set_format(type="torch", columns=cols)

# class weights on full train
y_full = train_final_df["label_bin"].to_numpy()
cw = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=y_full)
cw = torch.tensor(cw, dtype=torch.float32)

model_final = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

args_final = TrainingArguments(
    output_dir="roberta_final",
    eval_strategy="no",            # no tuning now
    save_strategy="no",
    learning_rate=float(best_cfg["lr"]),
    weight_decay=float(best_cfg["wd"]),
    num_train_epochs=int(best_cfg["epochs"]),
    per_device_train_batch_size=int(best_cfg["bs"]),
    per_device_eval_batch_size=32,
    fp16=True,
)

trainer_final = WeightedTrainer(
    class_weights=cw,
    sample_weights=None,           # optional: keep None for clean final fit
    model=model_final,
    args=args_final,
    train_dataset=train_final_hf,
    eval_dataset=dev_final_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_final.train()


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.592678
1000,0.433840
1500,0.310138
2000,0.177303
2500,0.104140


TrainOutput(global_step=2620, training_loss=0.31216092291679093, metrics={'train_runtime': 339.3173, 'train_samples_per_second': 123.41, 'train_steps_per_second': 7.721, 'total_flos': 2768766968876160.0, 'train_loss': 0.31216092291679093, 'epoch': 5.0})

In [19]:
# Final evaluation on official dev (once)

from sklearn.metrics import precision_score, recall_score, f1_score

pred = trainer_final.predict(dev_final_hf)
logits = torch.tensor(pred.predictions, dtype=torch.float32)
y_true = pred.label_ids
probs = torch.softmax(logits, dim=1).numpy()[:, 1]

# Report at threshold 0.5
y_05 = (probs >= 0.5).astype(int)
final_05 = {
    "threshold": 0.5,
    "precision": precision_score(y_true, y_05, zero_division=0),
    "recall": recall_score(y_true, y_05, zero_division=0),
    "f1": f1_score(y_true, y_05, zero_division=0),
}

# Report at fixed threshold from internal-dev tuning
y_bt = (probs >= BEST_T).astype(int)
final_best_t = {
    "threshold": BEST_T,
    "precision": precision_score(y_true, y_bt, zero_division=0),
    "recall": recall_score(y_true, y_bt, zero_division=0),
    "f1": f1_score(y_true, y_bt, zero_division=0),
}

final_05, final_best_t

({'threshold': 0.5,
  'precision': 0.6032608695652174,
  'recall': 0.5577889447236181,
  'f1': 0.5796344647519582},
 {'threshold': 0.3549999999999999,
  'precision': 0.5947368421052631,
  'recall': 0.5678391959798995,
  'f1': 0.5809768637532133})